# Day 1: Load H&M data and choose the window
Source: Kaggle *H&M Personalized Fashion Recommendations*, already converted to parquet in `Data/RAW/` and tracked by DVC (`Data/RAW.dvc`). Images are skipped.

## 1. Headers only (no rows loaded)

In [1]:
import pyarrow.parquet as pq
RAW = '../Data/RAW'
for f in ['transactions_train', 'customers', 'articles']:
    pf = pq.ParquetFile(f'{RAW}/{f}.parquet')
    print(f, f'{pf.metadata.num_rows:,} rows'); print('  ', pf.schema_arrow.names)

transactions_train 31,788,324 rows
   ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']
customers 1,371,980 rows
   ['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']
articles 105,542 rows
   ['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']


## 2. Column decisions

**Transactions: keep all 5** (`t_dat, customer_id, article_id, price, sales_channel_id`).
- `price` costs 4 bytes/row as float32. It drives price-affinity features and a revenue-weighted metric later. Keep.
- `sales_channel_id` costs 1 byte/row as int8. Online and in-store baskets behave differently. Keep.

**Customers: keep** `customer_id, age, club_member_status, fashion_news_frequency, FN, Active`.
- **Drop `postal_code`.** There are ~350k distinct values, so it is a quasi-identifier: postcode plus age can re-identify a person. It also adds no signal that a retailer could justify under GDPR. I wouldn't ship it.

**Articles: keep** `article_id, product_code` plus the readable category columns (`product_type_name, product_group_name, graphical_appearance_name, colour_group_name, perceived_colour_master_name, department_name, index_group_name, section_name, garment_group_name`).
- Drop the `*_no` / `*_code` twins: each maps 1:1 to its `*_name` column, so they're redundant.
- Drop `prod_name`: it is nearly 1:1 with `product_code`.
- **Drop `detail_desc` for V1.** Text embeddings are a V2 cold-start feature, not a Day 1 concern.

## 3. Dtypes
- `customer_id`: a 64-char hex string. Map it to a **dense int32 index** taken from `customers.parquet`, so every later notebook joins on the same integer. Save the lookup table once. (A pandas `category` would also shrink it, but each frame gets its own codes, so joins can silently misalign.)
- `article_id`: **int32 everywhere.** It's already int64 in both files and the leading zero only exists in image paths (`f'{id:010d}'`). Max ~959M, which fits in int32.
- `t_dat`: `datetime64`. `price`: float32. `sales_channel_id`: int8.

In [2]:
import pandas as pd
mb = lambda d: d.memory_usage(deep=True).sum() / 2**20

t = pd.read_parquet(f'{RAW}/transactions_train.parquet')
print(f'naive:   {len(t):,} rows, {mb(t):,.0f} MB'); print(t.dtypes)

naive:   31,788,324 rows, 3,456 MB
t_dat                   str
customer_id             str
article_id            int64
price               float64
sales_channel_id      int64
dtype: object


In [3]:
cust_ids = pd.read_parquet(f'{RAW}/customers.parquet', columns=['customer_id'])['customer_id']
t['customer_id'] = pd.Categorical(t['customer_id'], categories=cust_ids).codes.astype('int32')
assert (t['customer_id'] >= 0).all(), 'transaction customer missing from customers file'
t = t.astype({'article_id': 'int32', 'price': 'float32', 'sales_channel_id': 'int8'})
t['t_dat'] = pd.to_datetime(t['t_dat'])

art_ids = pd.read_parquet(f'{RAW}/articles.parquet', columns=['article_id'])['article_id']
assert t['article_id'].isin(art_ids).all(), 'article join would drop rows'
print(f'compact: {len(t):,} rows, {mb(t):,.0f} MB'); print(t.dtypes)

compact: 31,788,324 rows, 637 MB
t_dat               datetime64[us]
customer_id                  int32
article_id                   int32
price                      float32
sales_channel_id              int8
dtype: object


## 4. Counts

In [4]:
print('date range:', t.t_dat.min().date(), '->', t.t_dat.max().date())
print(f'rows {len(t):,} | customers {t.customer_id.nunique():,} | articles {t.article_id.nunique():,}')
# Kaggle's test week is Wed-Tue, so bin weeks ending Tuesday
weekly = t.groupby(pd.Grouper(key='t_dat', freq='W-TUE')).agg(
    rows=('article_id', 'size'), customers=('customer_id', 'nunique'), articles=('article_id', 'nunique'))
weekly.tail(16)

date range: 2018-09-20 -> 2020-09-22


rows 31,788,324 | customers 1,362,281 | articles 104,547


,rows,customers,articles
t_dat,,,
2020-06-09,310996,78193,21894
2020-06-16,298890,77306,21785
2020-06-23,549443,120190,23281
2020-06-30,517428,115992,22837
2020-07-07,337419,85648,21503
2020-07-14,286672,75996,20598
2020-07-21,289528,77277,20553
2020-07-28,299469,80591,21204
2020-08-04,319469,82802,20646


In [5]:
end = t.t_dat.max()
pd.DataFrame([{'weeks': n, **(lambda s: {'rows': len(s), 'customers': s.customer_id.nunique(),
               'articles': s.article_id.nunique()})(t[t.t_dat > end - pd.Timedelta(weeks=n)])}
              for n in (4, 8, 12, 16, 26, 52)]).set_index('weeks')

,rows,customers,articles
weeks,,,
4,1043390,233174,28297
8,2173875,378688,35576
12,3386963,489553,40772
16,5063720,594161,45774
26,8092617,744034,51316
52,14827645,992532,70768


## 5. Window: last 12 weeks (2020-07-01 to 2020-09-22)
- **Split room:** 10 train weeks, 1 validation week (09-09 to 09-15) and 1 test week (09-16 to 09-22).
- **Season:** the window is midsummer into early autumn, which matches the test week. It also starts just *after* the late-June clearance sale (weeks ending 06-23 and 06-30 are ~1.8x normal volume), so that one-off spike doesn't skew popularity.
- **Staleness:** ~41k active articles vs 105k in the catalogue. The model doesn't spend capacity on the ~60% that no longer sell.
- **Hardware:** ~3.4M rows is about 80 MB in RAM, so Day 8 training stays fast on a 16 GB laptop. If validation later shows sparse users hurting recall, try 16 weeks.

In [6]:
WEEKS = 12
win = t[t.t_dat > end - pd.Timedelta(weeks=WEEKS)].reset_index(drop=True)
import os; os.makedirs('../Data/processed', exist_ok=True)
win.to_parquet('../Data/processed/transactions_12w.parquet', index=False)
cust_ids.rename_axis('customer_idx').reset_index().to_parquet('../Data/processed/customer_ids.parquet', index=False)
print(win.t_dat.min().date(), '->', win.t_dat.max().date(), f'| {len(win):,} rows | {mb(win):,.0f} MB')

2020-07-01 -> 2020-09-22 | 3,386,963 rows | 68 MB


## Interview question
*Why might training on only recent weeks give a better recommender, and what do you lose?*

> Fashion demand drifts fast: last month's purchases reflect current stock, season and trends far better than purchases from two years ago, many of which are for articles that no longer exist. So a recent window trains on data shaped like the week you'll actually serve. The cost is history: customers who haven't bought recently become cold-start, and you lose long-term taste and yearly seasonality, such as what someone bought last autumn.